# Task 2: Modelling at Scale
**7006SCN Machine Learning and Big Data** | Weeks 3-4

Predict whether a taxi trip was paid in **cash** (1) or by **card** (0), using the cleaned data and the fitted pipeline from Task 1.

Four models, one from each of three families plus a free choice, each tuned with `CrossValidator` (3 folds). The tuning metric is **PR-AUC** (area under the precision-recall curve), because cash is only about 15% of trips and accuracy would hide mistakes on the minority class.

| Model | Family |
|---|---|
| LogisticRegression | Linear / generalised linear |
| RandomForestClassifier | Tree-based ensemble (bagging) |
| MultilayerPerceptronClassifier | Neural |
| GBTClassifier | Free choice: tree boosting |

**How to run:** run all cells from the top, in one session. If the Task 1 files are not on disk, the setup cell rebuilds them first.

## 1. Declaration

In [ ]:
# --- Student declaration ---
STUDENT_NAME  = "Sridhar Gude"
STUDENT_EMAIL = "gudes@coventry.ac.uk"
SID           = "17165640"
POOL_REF      = "No. 33 (Dataset Catalogue)"
DATASET_NAME  = "NYC TLC Yellow Taxi Trip Records, Jan-Jun 2024"
HEADER = f"{STUDENT_NAME} | {STUDENT_EMAIL} | {DATASET_NAME} | Pool ref {POOL_REF}"
print(HEADER)

## 2. Setup (loads Task 1 data and the fitted pipeline)

In [ ]:
# ---------- Setup: load the Task 1 data if it exists, otherwise rebuild it ----------
import os, sys, json, time, glob, urllib.request, subprocess
from functools import reduce
try:
    import pyspark
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
from pyspark.sql import SparkSession, DataFrame, functions as F
from pyspark import StorageLevel
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (SQLTransformer, Imputer, VectorAssembler, StandardScaler,
                                StringIndexer, OneHotEncoder)

RAW_DIR, OUT_DIR, MODEL_DIR, RES_DIR = "data/raw", "data/processed", "models", "results"
for d in (RAW_DIR, OUT_DIR, MODEL_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)
timings = {}

spark = (SparkSession.builder.appName("7006SCN_Task_NYC_Taxi_Payment").master("local[*]")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.session.timeZone", "UTC").getOrCreate())
sc = spark.sparkContext
CLUSTER = {"spark_version": spark.version, "master": sc.master, "cores": sc.defaultParallelism,
           "driver_memory": spark.conf.get("spark.driver.memory", "default"),
           "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
           "note": "local mode: one JVM, the driver also runs the tasks (no separate executors)"}
print(CLUSTER)

BASE  = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
FILES = [f"yellow_tripdata_2024-{m:02d}.parquet" for m in range(1, 7)]
COMMON = {"vendorid": "int", "tpep_pickup_datetime": "timestamp", "tpep_dropoff_datetime": "timestamp",
          "passenger_count": "double", "trip_distance": "double", "ratecodeid": "double",
          "store_and_fwd_flag": "string", "pulocationid": "int", "dolocationid": "int",
          "payment_type": "int", "fare_amount": "double", "extra": "double", "mta_tax": "double",
          "tip_amount": "double", "tolls_amount": "double", "improvement_surcharge": "double",
          "total_amount": "double", "congestion_surcharge": "double", "airport_fee": "double"}

def load_raw():
    parts = []
    for f in FILES:
        path = os.path.join(RAW_DIR, f)
        if not os.path.exists(path):
            print("downloading", f)
            urllib.request.urlretrieve(BASE + f, path)
        d = spark.read.parquet(path)
        d = d.toDF(*[c.lower() for c in d.columns])
        parts.append(d.select([F.col(c).cast(t).alias(c) for c, t in COMMON.items()]))
    return reduce(DataFrame.unionByName, parts)

feature_sql = """
SELECT *,
  (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0 AS trip_minutes,
  trip_distance / ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0) AS avg_speed_mph,
  fare_amount / trip_distance AS fare_per_mile,
  hour(tpep_pickup_datetime) AS pickup_hour,
  dayofweek(tpep_pickup_datetime) AS pickup_dow,
  CASE WHEN dayofweek(tpep_pickup_datetime) IN (1, 7) THEN 1.0 ELSE 0.0 END AS is_weekend,
  CASE WHEN passenger_count IS NULL THEN 1.0 ELSE 0.0 END AS passenger_missing,
  CASE WHEN congestion_surcharge IS NULL THEN 1.0 ELSE 0.0 END AS congestion_missing,
  CASE WHEN airport_fee IS NULL THEN 1.0 ELSE 0.0 END AS airport_missing,
  CAST(vendorid AS STRING) AS vendor_c,
  COALESCE(CAST(CAST(ratecodeid AS INT) AS STRING), 'unknown') AS rate_c,
  COALESCE(store_and_fwd_flag, 'unknown') AS flag_c,
  CAST(pulocationid AS STRING) AS pu_c,
  CAST(dolocationid AS STRING) AS do_c
FROM __THIS__
"""
to_impute = ["passenger_count", "congestion_surcharge", "airport_fee"]
num_cols  = ["passenger_count_imp", "congestion_surcharge_imp", "airport_fee_imp", "trip_distance",
             "fare_amount", "extra", "mta_tax", "tolls_amount", "improvement_surcharge", "trip_minutes",
             "avg_speed_mph", "fare_per_mile", "pickup_hour", "pickup_dow"]
flag_cols = ["is_weekend", "passenger_missing", "congestion_missing", "airport_missing"]
cat_cols  = ["vendor_c", "rate_c", "flag_c", "pu_c", "do_c"]

def build_pipeline():
    return Pipeline(stages=[
        SQLTransformer(statement=feature_sql),
        Imputer(strategy="median", inputCols=to_impute, outputCols=[c + "_imp" for c in to_impute]),
        VectorAssembler(inputCols=num_cols, outputCol="num_vec"),
        StandardScaler(inputCol="num_vec", outputCol="num_scaled", withMean=False, withStd=True),
        StringIndexer(inputCols=cat_cols, outputCols=[c + "_idx" for c in cat_cols], handleInvalid="keep"),
        OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols], outputCols=[c + "_ohe" for c in cat_cols],
                      handleInvalid="keep"),
        VectorAssembler(inputCols=["num_scaled"] + [c + "_ohe" for c in cat_cols] + flag_cols,
                        outputCol="features")])

def clean_rows(raw):
    pick, drop_ = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")
    minutes = (F.unix_timestamp(drop_) - F.unix_timestamp(pick)) / 60.0
    rules = [(pick >= F.to_timestamp(F.lit("2024-01-01"))) & (pick < F.to_timestamp(F.lit("2024-07-01"))),
             (minutes >= 1) & (minutes <= 180),
             (F.col("trip_distance") >= 0.1) & (F.col("trip_distance") <= 100),
             (F.col("fare_amount") > 0) & (F.col("fare_amount") <= 500),
             F.col("payment_type").isin(1, 2)]
    keep = reduce(lambda a, b: a & b, [F.coalesce(r, F.lit(False)) for r in rules])
    return (raw.filter(keep)
               .withColumn("is_cash", (F.col("payment_type") == 2).cast("int"))
               .withColumn("passenger_count", F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count")))
               .fillna(0, subset=["extra", "mta_tax", "tolls_amount", "improvement_surcharge"])
               .drop("tip_amount", "total_amount", "payment_type"))

train_path, test_path = os.path.join(OUT_DIR, "train.parquet"), os.path.join(OUT_DIR, "test.parquet")
model_path = os.path.join(MODEL_DIR, "prep_pipeline")
if os.path.exists(train_path) and os.path.exists(test_path) and os.path.exists(model_path):
    print("Found Task 1 data and pipeline on disk: loading them")
    train, test = spark.read.parquet(train_path), spark.read.parquet(test_path)
    prep_model = PipelineModel.load(model_path)
else:
    print("Task 1 files not found: rebuilding them (takes a few minutes)")
    t0 = time.time()
    clean = clean_rows(load_raw()).persist(StorageLevel.MEMORY_AND_DISK)
    train, test = clean.randomSplit([0.8, 0.2], seed=42)
    prep_model = build_pipeline().fit(train)
    train.write.mode("overwrite").parquet(train_path)
    test.write.mode("overwrite").parquet(test_path)
    prep_model.write().overwrite().save(model_path)
    train, test = spark.read.parquet(train_path), spark.read.parquet(test_path)
    print("rebuilt in", round(time.time() - t0, 1), "s")

## 3. Modelling data
Tuning four models with 3-fold cross-validation on 14 million rows would take too long on this machine, so I tune on a **3% stratified sample of the training split** and score on a **30% sample of the test split**. Both samples keep the class mix (Task 1 checked this on a 5% sample). The fitted Task 1 pipeline turns each row into the 551-feature vector.

In [ ]:
import numpy as np, pandas as pd
from pyspark.ml.functions import vector_to_array

TRAIN_FRACTION, TEST_FRACTION = 0.03, 0.30
train_s = train.sampleBy("is_cash", fractions={0: TRAIN_FRACTION, 1: TRAIN_FRACTION}, seed=42)
test_s  = test.sample(False, TEST_FRACTION, seed=7)

def to_model_frame(df):
    return (prep_model.transform(df)
              .select(F.col("is_cash").cast("double").alias("label"), "features", "pu_c"))

t0 = time.time()
train_p = to_model_frame(train_s).persist(StorageLevel.MEMORY_AND_DISK)
test_p  = to_model_frame(test_s).persist(StorageLevel.MEMORY_AND_DISK)
n_train, n_test = train_p.count(), test_p.count()
n_features = train_p.head()["features"].size
timings["prepare_model_frames_seconds"] = round(time.time() - t0, 1)

cash_train = train_p.agg(F.avg("label")).first()[0]
print(f"train rows: {n_train:,}   test rows: {n_test:,}   features: {n_features}")
print(f"cash share in training sample: {cash_train*100:.2f}%")
print("prepare time:", timings["prepare_model_frames_seconds"], "s")

## 4. The four models and their parameter grids
- **LogisticRegression**: searches `regParam` (0, 0.01, 0.1). The features are already scaled, so `standardization` is off.
- **RandomForest**: 30 trees, searches `maxDepth` (6, 10). Deeper trees can pick out zone and fare patterns but cost more time.
- **MultilayerPerceptron**: one hidden layer, searches the hidden size (16, 32).
- **GBTClassifier**: 20 boosting rounds, searches `maxDepth` (3, 5). Shallow trees fitted one after another.

In [ ]:
from pyspark.ml.classification import (LogisticRegression, RandomForestClassifier, GBTClassifier,
                                       MultilayerPerceptronClassifier)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr  = LogisticRegression(maxIter=40, standardization=False)
rf  = RandomForestClassifier(numTrees=30, maxBins=32, seed=42)
gbt = GBTClassifier(maxIter=20, seed=42)
mlp = MultilayerPerceptronClassifier(maxIter=40, blockSize=128, seed=42)

candidates = [
    ("LogisticRegression",   "Linear / generalised linear", lr,
        ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.01, 0.1]).build()),
    ("RandomForest",         "Tree-based ensemble (bagging)", rf,
        ParamGridBuilder().addGrid(rf.maxDepth, [6, 10]).build()),
    ("MultilayerPerceptron", "Neural network", mlp,
        ParamGridBuilder().addGrid(mlp.layers, [[n_features, 16, 2], [n_features, 32, 2]]).build()),
    ("GBTClassifier",        "Free choice: tree boosting", gbt,
        ParamGridBuilder().addGrid(gbt.maxDepth, [3, 5]).build()),
]
GRID_TEXT = {
    "LogisticRegression": "regParam in {0, 0.01, 0.1}",
    "RandomForest": "maxDepth in {6, 10}; 30 trees",
    "MultilayerPerceptron": "hidden units in {16, 32}",
    "GBTClassifier": "maxDepth in {3, 5}; 20 rounds",
}
NUM_FOLDS = 3
tuning_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                                 metricName="areaUnderPR")

## 5. Train and tune (CrossValidator)
Training time covers the whole cross-validation plus the final refit on the full training sample. This cell is the slow one (expect 20 to 60 minutes).

In [ ]:
trained = {}
for name, family, est, grid in candidates:
    cv = CrossValidator(estimator=est, estimatorParamMaps=grid, evaluator=tuning_evaluator,
                        numFolds=NUM_FOLDS, seed=42, parallelism=1)
    t0 = time.time()
    cv_model = cv.fit(train_p)
    secs = round(time.time() - t0, 1)
    best_i = int(np.argmax(cv_model.avgMetrics))
    best_params = {p.name: (list(v) if isinstance(v, (list, tuple)) else v) for p, v in grid[best_i].items()}
    trained[name] = {"model": cv_model.bestModel, "family": family, "seconds": secs,
                     "cv_pr_auc_per_setting": [round(float(m), 4) for m in cv_model.avgMetrics],
                     "best_params": best_params, "grid_size": len(grid)}
    cv_model.bestModel.write().overwrite().save(os.path.join(MODEL_DIR, name))
    print(f"{name:22s} | {secs:8.1f} s | best {best_params} | CV PR-AUC {max(cv_model.avgMetrics):.4f}")

## 6. Evaluate on the test sample
Metrics are for the **cash** class (label 1) at the default 0.5 threshold, plus ROC-AUC and PR-AUC, which do not depend on a threshold.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

roc_eval = BinaryClassificationEvaluator(metricName="areaUnderROC")
pr_eval  = BinaryClassificationEvaluator(metricName="areaUnderPR")

results, curves, conf = {}, {}, {}
for name, info in trained.items():
    t0 = time.time()
    preds = info["model"].transform(test_p).select("label", "prediction", "rawPrediction", "probability").persist()
    roc, pr = roc_eval.evaluate(preds), pr_eval.evaluate(preds)
    cm = {(int(r["label"]), int(r["prediction"])): int(r["count"])
          for r in preds.groupBy("label", "prediction").count().collect()}
    tp, fp, fn, tn = cm.get((1, 1), 0), cm.get((0, 1), 0), cm.get((1, 0), 0), cm.get((0, 0), 0)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy  = (tp + tn) / (tp + fp + fn + tn)
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    curves[name] = (preds.sample(False, min(1.0, 100000 / n_test), seed=1)
                         .select("label", vector_to_array("probability")[1].alias("score")).toPandas())
    conf[name] = np.array([[tn, fp], [fn, tp]])
    results[name] = {"accuracy": round(accuracy, 4), "precision_cash": round(precision, 4),
                     "recall_cash": round(recall, 4), "f1_cash": round(f1, 4),
                     "roc_auc": round(roc, 4), "pr_auc": round(pr, 4),
                     "balanced_accuracy": round((recall + specificity) / 2, 4),
                     "scoring_seconds": round(time.time() - t0, 1)}
    preds.unpersist()

metrics_table = pd.DataFrame(results).T
metrics_table.insert(0, "train_seconds", [trained[n]["seconds"] for n in metrics_table.index])
metrics_table

## EP2: Evidence pack
Take **one screenshot** of the output of the cell below. It prints all four training times and best parameters in one loop, then draws the four confusion matrices with the ROC and PR curves. (Curves use a 100,000-row sample of the test sample; the table above uses the whole test sample.)

In [ ]:
import matplotlib.pyplot as plt

print("=" * 100); print(HEADER); print("=" * 100)
for name, info in trained.items():
    print(f"{name:22s} | CV folds {NUM_FOLDS} | train {info['seconds']:8.1f} s | best params {info['best_params']}")

fig = plt.figure(figsize=(18, 9))
gs = fig.add_gridspec(2, 4)
names = list(trained)
for i, name in enumerate(names):
    ax = fig.add_subplot(gs[0, i]); m = conf[name]
    ax.imshow(m, cmap="Blues")
    for (r, c), v in np.ndenumerate(m):
        ax.text(c, r, f"{v:,}", ha="center", va="center", fontsize=9,
                color="white" if v > m.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["card", "cash"]); ax.set_yticklabels(["card", "cash"])
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title(name, fontsize=10)

ax = fig.add_subplot(gs[1, 0:2])
for name in names:
    d = curves[name]; fpr, tpr, _ = roc_curve(d["label"], d["score"])
    ax.plot(fpr, tpr, label=f"{name} (AUC {results[name]['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_title("ROC curves")
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, 2:4])
for name in names:
    d = curves[name]; p, r, _ = precision_recall_curve(d["label"], d["score"])
    ax.plot(r, p, label=f"{name} (AP {results[name]['pr_auc']:.3f})")
ax.set_title("Precision-recall curves"); ax.set_xlabel("recall (cash)"); ax.set_ylabel("precision (cash)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. results/task2.json

In [ ]:
algorithms = []
for name, info in trained.items():
    algorithms.append({"name": name, "family": info["family"], "param_grid": GRID_TEXT[name],
                       "grid_size": info["grid_size"], "cv_folds": NUM_FOLDS,
                       "cv_pr_auc_per_setting": info["cv_pr_auc_per_setting"],
                       "best_params": info["best_params"], "training_seconds": info["seconds"],
                       "metrics": results[name]})

task2 = {"student_name": STUDENT_NAME, "sid": SID, "pool_reference": POOL_REF,
         "tuning_metric": "areaUnderPR", "cv_folds": NUM_FOLDS,
         "train_fraction": TRAIN_FRACTION, "test_fraction": TEST_FRACTION,
         "train_rows": n_train, "test_rows": n_test, "feature_vector_length": n_features,
         "cash_share_train_pct": round(cash_train * 100, 2),
         "families": sorted({i["family"] for i in algorithms}),
         "algorithms": algorithms, "cluster_config": CLUSTER, "timings_seconds": timings}
with open(os.path.join(RES_DIR, "task2.json"), "w") as fh:
    json.dump(task2, fh, indent=2)

best_name = max(results, key=lambda n: results[n]["pr_auc"])
print("saved results/task2.json")
print("highest PR-AUC:", best_name, results[best_name]["pr_auc"])
print("\nBest parameters (keep this, Task 3 can use it):")
print({n: trained[n]["best_params"] for n in trained})